In [1]:
import sys 

sys.path.append('../')
from corpus.jsinV3_attn_cue_multi_source import jsinV3_attn_cue_multi_source
import src.audio_attention_transforms as aat
import src.audio_transforms as at


/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np 
import torch 
import yaml

In [45]:
config_path = '../config/attentional_cue/attn_cue_match_target_speech_distractor_only.yaml'
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

In [46]:
config['data']

{'num_words': 794,
 'multi_distractor': True,
 'corpus': {'n_talkers': [1, 5],
  'with_audioset': False,
  'root': '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/'},
 'audio': {'rep_type': 'cochlea_filt',
  'rep_kwargs': {'sr': 20000,
   'env_sr': 8000,
   'n_channels': 40,
   'low_lim': 40,
   'use_pad': True,
   'rep_on_gpu': False,
   'env_extraction_type': 'Half-wave Rectification',
   'downsampling_type': 'TorchTransformsResample',
   'downsampling_kwargs': {'lowpass_filter_width': 64,
    'rolloff': 0.9475937167399596,
    'resampling_method': 'kaiser_window',
    'beta': 14.769656459379492}},
  'compression_type': 'coch_p3',
  'compression_kwargs': {'scale': 1,
   'offset': 1e-07,
   'clip_value': 5,
   'power': 0.3}},
 'loader': {'batch_size': 128}}

In [47]:
audio_transforms = aat.AudioCompose([
    aat.AudioToTensor(),
    aat.RMSNormalizeForegroundAndBackground(rms_level=0.1),
    aat.CombineWithRandomDBSNR(low_snr=config['noise_kwargs']['low_snr'], high_snr=config['noise_kwargs']['high_snr']),
    aat.RMSNormalizeMixtureAndMatchCueLevel(rms_level=0.1),
    aat.UnsqueezeAudio(dim=0),
])
# these transforms take foreground, background as input 
bg_combine_transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
                at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set distractors to same level for matched cue level training  
                # at.CombineWithRandomDBSNR(low_snr=config['noise_kwargs']['low_snr'], high_snr=config['noise_kwargs']['high_snr']),
                at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
                # at.UnsqueezeAudio(dim=0),
            ])



In [48]:
coch_transform = aat.AudioToAudioRepresentation(**config['data']['audio'])


In [49]:
dataset = jsinV3_attn_cue_multi_source(**config['data']['corpus'],
                                       mode='train',
                                       noise_only=False,
                                       transform=[audio_transforms, bg_combine_transforms], demo=True)

In [50]:
dataset[211]

(array([-0.08628907, -0.09155136, -0.1048617 , ..., -0.16419941,
        -0.13410501, -0.11710397], dtype=float32),
 array([ 0.00326168,  0.05425015,  0.08945551, ..., -0.00494109,
        -0.03941556, -0.0478887 ], dtype=float32),
 None,
 tensor([[ 0.0072,  0.0197,  0.0273,  ..., -0.0025, -0.0083, -0.0088]]),
 tensor([[ 0.0117,  0.0161, -0.0126,  ...,  0.0854,  0.0947,  0.1026]]),
 49)

In [51]:
foreground, background, noise, signal, fg_cue, fg_target = dataset[211]

In [52]:
from IPython.display import Audio

In [53]:
Audio(foreground, rate=20000, normalize=True)

In [54]:
Audio(background, rate=20000, normalize=True)

In [55]:
Audio(noise, rate=20000, normalize=True)

ValueError: No audio data found. Expecting filename, url, or data.

In [56]:
Audio(fg_cue, rate=20000, normalize=False)

In [57]:
Audio(signal, rate=20000, normalize=False)



In [25]:
%matplotlib inline
import matplotlib.pyplot as plt

In [26]:
# plt.imshow(signal.squeeze().numpy(), aspect='auto')

In [ ]:
plt.imshow(fg_cue.squeeze().numpy(), aspect='auto')